# 2026-cv-competition: детекция (52 класса)

EDA → данные → обучение (YOLOv8 + RT-DETR) → WBF → `sample_submission.csv`.

**Локально:** репозиторий и каталог `2026-cv-competition/` с `sample_submission.csv`. Зависимости: `uv pip install torch ultralytics ensemble-boxes pandas numpy matplotlib seaborn pillow pyyaml scikit-learn ipykernel jupyter`. Запуск: `uv run jupyter notebook solution.ipynb`.

**Kaggle:** GPU; в наборе — `train/`, `test/`, `sample_submission.csv`. У закрытых заданий датасет подключают через вкладку **Competition**. Если `/kaggle/input` пустой при включённом Internet — см. вторую кодовую ячейку (`kagglehub`, env `KAGGLE_COMP`, по умолчанию `2026-cv-competition`).

Сабмит: `/kaggle/working/sample_submission.csv`. Основные env — в ячейках обучения и инференса (`TRAIN_*`, веса, `WBF_*`, `CONF_*`, `DO_SWEEP`).

RNG **993**.


In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd

RNG_SEED = 993

os.environ["PYTHONHASHSEED"] = str(RNG_SEED)
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)

import torch

torch.manual_seed(RNG_SEED)
torch.cuda.manual_seed_all(RNG_SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())

In [ ]:
import os
import subprocess
import sys
import zipfile
from pathlib import Path

KAGGLE_COMP_SLUG = "2026-cv-competition"


def _is_kaggle() -> bool:
    return bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))


def _pip_kaggle(packages: list[str]) -> None:
    if not _is_kaggle():
        return
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])


def _try_yolo_layout(comp: Path) -> tuple[Path, Path, Path] | None:
    """Data-страница: train/images + train/labels + test; ручной датасет: train + labels в корне; локальный zip: train/train/...."""
    for ti_rel, tl_rel, te_rel in (
        ("train/images", "train/labels", "test/images"),
        ("train/images", "train/labels", "test"),
        ("train", "labels", "test/images"),
        ("train", "labels", "test"),
        ("train/train/images", "train/train/labels", "test/test/images"),
    ):
        ti, tl, te = comp / ti_rel, comp / tl_rel, comp / te_rel
        if ti.is_dir() and tl.is_dir() and te.is_dir():
            return ti, tl, te
    return None


def _datasets_under_input(input_root: Path) -> list[tuple[Path, Path]]:
    """Список (папка_соревнования, sample_submission.csv)."""
    found: list[tuple[Path, Path]] = []
    if not input_root.is_dir():
        return found

    seen: set[Path] = set()

    for csv in sorted(input_root.rglob("sample_submission.csv")):
        comp = csv.parent.resolve()
        if comp in seen:
            continue
        if _try_yolo_layout(comp) is not None:
            seen.add(comp)
            found.append((comp, csv.resolve()))

    found.sort(key=lambda p: len(p[0].parts))
    return found


def _download_via_kagglehub() -> tuple[Path, Path]:
    """Если Input пустой — качаем соревнование через kagglehub (slug: KAGGLE_COMP или 2026-cv-competition)."""
    slug = (os.environ.get("KAGGLE_COMP") or KAGGLE_COMP_SLUG).strip()
    _pip_kaggle(["kagglehub"])

    import kagglehub

    folder = Path(kagglehub.competition_download(slug)).resolve()

    csv_primary = folder / "sample_submission.csv"
    if csv_primary.is_file():
        ys = _try_yolo_layout(folder)
        if ys is not None:
            return folder, csv_primary.resolve()

    for csv in sorted(folder.rglob("sample_submission.csv")):
        comp = csv.parent.resolve()
        if _try_yolo_layout(comp) is not None:
            return comp, csv.resolve()

    raise FileNotFoundError(
        "kagglehub скачал архив в "
        + str(folder)
        + ", но нужных папок train/... и test/... там не нашлось."
    )




def _extract_local_comp_zip(base: Path) -> None:
    """Локально: при отсутствии папки датасета — распаковать zip соревнования в корень поиска."""
    comp_dir = base / "2026-cv-competition"
    if comp_dir.is_dir() and (comp_dir / "sample_submission.csv").is_file():
        return

    candidates = [base / "2026-cv-competition.zip", *sorted(base.glob("*.zip"))]
    seen: set[Path] = set()

    for z in candidates:
        z = z.resolve()
        if z in seen or not z.is_file():
            continue
        seen.add(z)
        try:
            with zipfile.ZipFile(z) as zf:
                names = zf.namelist()
                has_sample = any(name.endswith("sample_submission.csv") for name in names)
                has_train = any("train/" in name for name in names)
                has_test = any("test/" in name for name in names)
                if not (has_sample and has_train and has_test):
                    continue
                zf.extractall(base)
                return
        except zipfile.BadZipFile:
            continue

def _find_competition_and_root() -> tuple[Path, Path, Path, Path, Path, Path]:
    """ROOT, COMP, TRAIN_IMG, TRAIN_LBL, TEST_IMG, SAMPLE_SUB."""
    here = Path.cwd().resolve()

    if not _is_kaggle():
        for p in [here, *here.parents]:
            _extract_local_comp_zip(p)
            comp = p / "2026-cv-competition"
            csv_path = comp / "sample_submission.csv"
            if not csv_path.is_file():
                continue
            ys = _try_yolo_layout(comp.resolve())
            if ys is None:
                raise FileNotFoundError(
                    "Локально: есть sample_submission.csv, но не совпало дерево: "
                    "train/images + train/labels + test, либо train + labels в корне, либо train/train/...."
                )
            ti, tl, te = ys
            return p, comp.resolve(), ti, tl, te, csv_path.resolve()
        raise FileNotFoundError(
            "Не найден 2026-cv-competition/sample_submission.csv — "
            "положите данные рядом с репозиторием или используйте Kaggle."
        )

    _pip_kaggle(["ultralytics"])
    os.environ.setdefault("KAGGLE_COMP", KAGGLE_COMP_SLUG)

    root = Path("/kaggle/working")
    input_root = Path("/kaggle/input")

    pairs = _datasets_under_input(input_root) if input_root.is_dir() else []

    if not pairs:
        comp, samp = _download_via_kagglehub()
    else:
        comp, samp = pairs[0]

    ys = _try_yolo_layout(comp)
    if ys is None:
        raise FileNotFoundError(f"Не удалось определить папки train/test относительно {comp}")
    ti, tl, te = ys

    return root, comp, ti, tl, te, samp


ROOT, COMP, TRAIN_IMG, TRAIN_LBL, TEST_IMG, SAMPLE_SUB = _find_competition_and_root()

RUNS = ROOT / "runs"
RUNS.mkdir(parents=True, exist_ok=True)

for pth in (TRAIN_IMG, TRAIN_LBL, TEST_IMG):
    assert pth.is_dir(), pth
assert SAMPLE_SUB.is_file(), SAMPLE_SUB
print(
    "ROOT:", ROOT,
    "| COMP:", COMP,
    "| kaggle:", _is_kaggle(),
    "| SAMPLE_SUB:", SAMPLE_SUB,
    "| KAGGLE_COMP:", os.environ.get("KAGGLE_COMP") or (KAGGLE_COMP_SLUG if _is_kaggle() else "n/a"),
)


## 1. EDA

Классы и дисбаланс, плотность разметки, размеры кадров, геометрия боксов (площадь и аспект в YOLO-нормализации), примеры разметки.

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Rectangle
from PIL import Image

sns.set_theme(style="whitegrid", context="notebook")

label_files = sorted(TRAIN_LBL.glob("*.txt"))
class_counts = Counter()
objs_per_image = []

for lf in label_files:
    lines = [ln for ln in lf.read_text(encoding="utf-8").splitlines() if ln.strip()]
    objs_per_image.append(len(lines))
    for ln in lines:
        class_counts[int(ln.split()[0])] += 1

n_classes = max(class_counts) + 1 if class_counts else 0
print("Размеченных файлов:", len(label_files))
print("Число классов (max id + 1):", n_classes)
print("Всего объектов:", sum(class_counts.values()))
print("Объектов на картинку: min/median/max", np.min(objs_per_image), np.median(objs_per_image), np.max(objs_per_image))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cls_ids = sorted(class_counts)
axes[0].bar(cls_ids, [class_counts[c] for c in cls_ids], color="steelblue")
axes[0].set_title("Частота классов (все боксы)")
axes[0].set_xlabel("class_id")
axes[0].set_ylabel("count")

axes[1].hist(objs_per_image, bins=30, color="coral", edgecolor="white")
axes[1].set_title("Число объектов на изображение")
axes[1].set_xlabel("objects per image")
plt.tight_layout()
plt.show()

In [ ]:
# размеры кадров train/test; геометрия боксов (нормализованная YOLO)

train_imgs = sorted(TRAIN_IMG.glob("*.jpg")) + sorted(TRAIN_IMG.glob("*.jpeg")) + sorted(TRAIN_IMG.glob("*.png"))
test_imgs = sorted(TEST_IMG.glob("*.jpg")) + sorted(TEST_IMG.glob("*.jpeg")) + sorted(TEST_IMG.glob("*.png"))
label_stems = {lf.stem for lf in label_files}
have_lbl_no_img = sorted(label_stems - {p.stem for p in train_imgs})
have_img_no_lbl = sorted({p.stem for p in train_imgs if p.stem not in label_stems})
print("train images:", len(train_imgs), "| test:", len(test_imgs))
print("файлов labels:", len(label_files), "| без пары-картинки:", len(have_lbl_no_img))
print("картинок без label-текста:", len(have_img_no_lbl))

if train_imgs:
    ws = []
    hs = []
    for p in train_imgs:
        with Image.open(p) as im:
            w, h = im.size
        ws.append(w)
        hs.append(h)
    ws = np.asarray(ws, dtype=np.float64)
    hs = np.asarray(hs, dtype=np.float64)
    asp = ws / np.maximum(hs, 1.0)
    print(
        "train W px  q05/q50/q95:", np.percentile(ws, [5, 50, 95]).round(2),
        "| H px q05/q50/q95:", np.percentile(hs, [5, 50, 95]).round(2),
        "| aspect W/H q50:", round(float(np.quantile(asp, 0.5)), 3),
    )

box_w = []
box_h = []
box_area = []
for lf in label_files:
    text = lf.read_text(encoding="utf-8").splitlines()
    for ln in text:
        if not ln.strip():
            continue
        parts = ln.split()
        bw, bh = map(float, parts[3:5])
        box_w.append(bw)
        box_h.append(bh)
        box_area.append(bw * bh)

if box_area:
    box_w = np.asarray(box_w)
    box_h = np.asarray(box_h)
    box_area = np.asarray(box_area)
    ar_boxes = np.divide(box_w, np.maximum(box_h, 1e-6))
    print(
        "норм. ширина/высота бокса  median:",
        round(float(np.median(box_w)), 4),
        round(float(np.median(box_h)), 4),
        "| медиана площади (bw·bh):",
        round(float(np.median(box_area)), 6),
    )
    print("апспект бокса W/H median / q95:", round(float(np.median(ar_boxes)), 3), "/", round(float(np.quantile(ar_boxes, 0.95)), 3))

top_cls = sorted(class_counts.items(), key=lambda x: -x[1])[:12]
print(pd.DataFrame(top_cls, columns=["class_id", "n_boxes"]).to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
if train_imgs:
    axes[0].hist(ws, bins=min(48, len(np.unique(ws))), color="steelblue", edgecolor="white")
axes[0].set_title("Ширина изображений, px")

if len(box_area):
    axes[1].hist(box_area * 100.0, bins=52, color="seagreen", edgecolor="white")
    axes[1].set_xlabel("площадь бокса, % кадра (100·bw·bh)")
axes[1].set_title("Распределение площади бокса")

if len(box_area):
    axes[2].hist(np.clip(ar_boxes, 0.02, 8.0), bins=42, color="coral", edgecolor="white")
    axes[2].set_xlabel("соотношение W/H (обрезка для читаемости)")
axes[2].set_title("Форма боксов")
plt.tight_layout()
plt.show()

In [ ]:
def yolo_norm_to_xyxy(cx, cy, w, h):
    return cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2


def show_annotated(img_path: Path):
    img = Image.open(img_path).convert("RGB")
    w0, h0 = img.size
    lf = TRAIN_LBL / (img_path.stem + ".txt")
    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    ax.imshow(img)
    ax.axis("off")
    if lf.is_file():
        for ln in lf.read_text(encoding="utf-8").splitlines():
            if not ln.strip():
                continue
            p = ln.split()
            cid = int(p[0])
            cx, cy, bw, bh = map(float, p[1:5])
            x1, y1, x2, y2 = yolo_norm_to_xyxy(cx, cy, bw, bh)
            x1, x2 = x1 * w0, x2 * w0
            y1, y2 = y1 * h0, y2 * h0
            rect = Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="lime", linewidth=2)
            ax.add_patch(rect)
            ax.text(x1, max(0, y1 - 4), str(cid), color="yellow", fontsize=8, fontweight="bold")
    plt.title(img_path.name)
    plt.show()


demo_imgs = sorted(TRAIN_IMG.glob("*.jpg"))[:3]
for ip in demo_imgs:
    show_annotated(ip)

## 2. Подготовка данных для YOLO (train/val split)

Списки абсолютных путей к изображениям — Ultralytics подставит `labels/` вместо `images/` для разметки.

In [ ]:
from sklearn.model_selection import train_test_split
import yaml

all_images = sorted(TRAIN_IMG.glob("*.jpg"))
train_paths, val_paths = train_test_split(
    all_images,
    test_size=0.15,
    random_state=RNG_SEED,
    shuffle=True,
)

split_dir = RUNS / "yolo_split"
split_dir.mkdir(parents=True, exist_ok=True)
train_txt = split_dir / "train.txt"
val_txt = split_dir / "val.txt"

train_txt.write_text("\n".join(str(p.resolve()) for p in train_paths), encoding="utf-8")
val_txt.write_text("\n".join(str(p.resolve()) for p in val_paths), encoding="utf-8")

names = [str(i) for i in range(52)]
data_yaml = {
    "train": str(train_txt.resolve()),
    "val": str(val_txt.resolve()),
    "nc": 52,
    "names": names,
}
yaml_path = split_dir / "dataset.yaml"
yaml_path.write_text(yaml.safe_dump(data_yaml, sort_keys=False, allow_unicode=True), encoding="utf-8")

print("train:", len(train_paths), "val:", len(val_paths))
print("yaml:", yaml_path)

## 3. Обучение

YOLOv8 и RT-DETR подряд, общий `dataset.yaml`.

Env: `TRAIN_IMGSZ`, `TRAIN_EPOCHS`, `TRAIN_BATCH`, `PATIENCE`, `YOLO_WEIGHTS`, `RT_WEIGHTS`, `RUN_YOLO`, `RUN_RT`.


In [ ]:
import subprocess
import sys

from ultralytics import YOLO

_is_kaggle = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))


def _pip_if_kaggle(pkgs: list[str]) -> None:
    if not _is_kaggle:
        return
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])


_pip_if_kaggle(["ensemble-boxes"])


def _avg_closed_epochs(default_epochs: int = 40) -> int:
    """Average completed epochs from past runs/detect/*/results.csv."""
    hist = []
    for rf in sorted((RUNS / "detect").glob("*/results.csv")):
        try:
            n = len(pd.read_csv(rf))
        except Exception:
            continue
        if n > 0:
            hist.append(int(n))
    if not hist:
        return default_epochs
    return max(1, int(round(sum(hist) / len(hist))))


TRAIN_IMGSZ = int(os.environ.get("TRAIN_IMGSZ", "640"))
AVG_EPOCHS = _avg_closed_epochs(default_epochs=40)
TRAIN_EPOCHS = int(os.environ.get("TRAIN_EPOCHS", "30"))
TRAIN_BATCH = int(os.environ.get("TRAIN_BATCH", "8" if _is_kaggle else "4"))
PATIENCE = int(os.environ.get("PATIENCE", "15"))

YOLO_WEIGHTS = os.environ.get("YOLO_WEIGHTS", "yolov8m.pt")
RT_WEIGHTS = os.environ.get("RT_WEIGHTS", "rtdetr-l.pt")
RUN_YOLO = os.environ.get("RUN_YOLO", "yolov8m_dual")
RUN_RT = os.environ.get("RUN_RT", "rtdetr_l_dual")

print(
    "dual-train | imgsz", TRAIN_IMGSZ,
    "epochs", TRAIN_EPOCHS,
    "(avg closed:", AVG_EPOCHS, ")",
    "batch", TRAIN_BATCH,
    "yolo", YOLO_WEIGHTS,
    "rtdetr", RT_WEIGHTS,
)

m1 = YOLO(YOLO_WEIGHTS)
m1.train(
    data=str(yaml_path),
    epochs=TRAIN_EPOCHS,
    imgsz=TRAIN_IMGSZ,
    batch=TRAIN_BATCH,
    seed=RNG_SEED,
    deterministic=True,
    project=str(RUNS / "detect"),
    name=RUN_YOLO,
    exist_ok=True,
    workers=int(os.environ.get("WORKERS", "2" if (_is_kaggle and torch.cuda.is_available()) else "0")),
    patience=PATIENCE,
    cos_lr=True,
)

m2 = YOLO(RT_WEIGHTS)
m2.train(
    data=str(yaml_path),
    epochs=TRAIN_EPOCHS,
    imgsz=TRAIN_IMGSZ,
    batch=max(1, TRAIN_BATCH // 2),
    seed=RNG_SEED,
    deterministic=True,
    project=str(RUNS / "detect"),
    name=RUN_RT,
    exist_ok=True,
    workers=int(os.environ.get("WORKERS", "2" if (_is_kaggle and torch.cuda.is_available()) else "0")),
    patience=PATIENCE,
    cos_lr=True,
)

YOLO_BEST = RUNS / "detect" / RUN_YOLO / "weights" / "best.pt"
RT_BEST = RUNS / "detect" / RUN_RT / "weights" / "best.pt"
assert YOLO_BEST.is_file(), YOLO_BEST
assert RT_BEST.is_file(), RT_BEST
print("best yolo:", YOLO_BEST)
print("best rtdetr:", RT_BEST)


## 4. Инференс, WBF

`ensemble-boxes`: слияние боксов двух моделей. `WBF_IOU` — порог объединения.

Env: `IMGSZ_PRED`, `PRED_CONF`, `WBF_IOU`, `WBF_SWEEP`, `RUN_YOLO`, `RUN_RT`.


In [ ]:
import subprocess
import sys

import numpy as np
from ultralytics import YOLO


def _ensure_ensemble_boxes() -> None:
    try:
        from ensemble_boxes import weighted_boxes_fusion  # noqa: F401
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ensemble-boxes"])


_ensure_ensemble_boxes()
from ensemble_boxes import weighted_boxes_fusion

_is_kaggle = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
RUN_YOLO = os.environ.get("RUN_YOLO", "yolov8m_dual")
RUN_RT = os.environ.get("RUN_RT", "rtdetr_l_dual")

YOLO_BEST = globals().get("YOLO_BEST") or (RUNS / "detect" / RUN_YOLO / "weights" / "best.pt")
RT_BEST = globals().get("RT_BEST") or (RUNS / "detect" / RUN_RT / "weights" / "best.pt")
assert YOLO_BEST.is_file(), YOLO_BEST
assert RT_BEST.is_file(), RT_BEST

IMGSZ_PRED = int(os.environ.get("IMGSZ_PRED", str(globals().get("TRAIN_IMGSZ", 640))))
PRED_CONF = float(os.environ.get("PRED_CONF", "0.005"))
WBF_IOU = float(os.environ.get("WBF_IOU", "0.55"))

mdl_y = YOLO(str(YOLO_BEST))
mdl_r = YOLO(str(RT_BEST))

pred_y = list(
    mdl_y.predict(
        source=str(TEST_IMG),
        imgsz=IMGSZ_PRED,
        conf=PRED_CONF,
        iou=0.7,
        augment=os.environ.get("AUGMENT", "1") == "1",
        save=False,
        verbose=False,
        stream=True,
    )
)
pred_r = list(
    mdl_r.predict(
        source=str(TEST_IMG),
        imgsz=IMGSZ_PRED,
        conf=PRED_CONF,
        iou=0.7,
        augment=os.environ.get("AUGMENT", "1") == "1",
        save=False,
        verbose=False,
        stream=True,
    )
)

by_stem_r = {Path(r.path).stem: r for r in pred_r}


def _rb_to_np(r):
    if r.boxes is None or len(r.boxes) == 0:
        return (
            np.zeros((0, 4), dtype=np.float32),
            np.zeros((0,), dtype=np.float32),
            np.zeros((0,), dtype=np.int64),
        )
    return (
        r.boxes.xyxy.cpu().numpy().astype(np.float32),
        r.boxes.conf.cpu().numpy().astype(np.float32),
        r.boxes.cls.cpu().numpy().astype(np.int64),
    )


def _fuse_one(ry, rr, wbf_iou: float):
    h0, w0 = ry.orig_shape
    b1, s1, l1 = _rb_to_np(ry)
    b2, s2, l2 = _rb_to_np(rr)
    if b1.shape[0] == 0:
        return b2, s2, l2, (h0, w0)
    if b2.shape[0] == 0:
        return b1, s1, l1, (h0, w0)

    a = b1.astype(np.float64)
    b = b2.astype(np.float64)
    aa = np.stack(
        [
            np.clip(a[:, 0] / w0, 0.0, 1.0),
            np.clip(a[:, 1] / h0, 0.0, 1.0),
            np.clip(a[:, 2] / w0, 0.0, 1.0),
            np.clip(a[:, 3] / h0, 0.0, 1.0),
        ],
        axis=1,
    )
    bb = np.stack(
        [
            np.clip(b[:, 0] / w0, 0.0, 1.0),
            np.clip(b[:, 1] / h0, 0.0, 1.0),
            np.clip(b[:, 2] / w0, 0.0, 1.0),
            np.clip(b[:, 3] / h0, 0.0, 1.0),
        ],
        axis=1,
    )

    bx, sc, lb = weighted_boxes_fusion(
        [aa, bb],
        [s1, s2],
        [l1.astype(np.int32), l2.astype(np.int32)],
        weights=[1.0, 1.0],
        iou_thr=wbf_iou,
        skip_box_thr=1e-4,
    )
    fb = np.asarray(bx, dtype=np.float32)
    fs = np.asarray(sc, dtype=np.float32)
    fl = np.asarray(lb, dtype=np.int64)
    fb[:, 0] *= w0
    fb[:, 2] *= w0
    fb[:, 1] *= h0
    fb[:, 3] *= h0
    return fb, fs, fl, (h0, w0)


fused_by_wbf: dict[float, dict[str, dict]] = {}
for wbf_iou in [float(x.strip()) for x in os.environ.get("WBF_SWEEP", str(WBF_IOU)).split(",") if x.strip()]:
    out = {}
    for ry in pred_y:
        stem = Path(ry.path).stem
        rr = by_stem_r.get(stem)
        if rr is None:
            continue
        xyxy, conf, cls, orig = _fuse_one(ry, rr, wbf_iou)
        out[stem] = {"xyxy": xyxy, "conf": conf, "cls": cls, "orig_shape": orig}
    fused_by_wbf[wbf_iou] = out
    print("WBF iou", wbf_iou, "images", len(out))

results = fused_by_wbf[WBF_IOU]
print("using WBF_IOU", WBF_IOU, "| imgsz", IMGSZ_PRED, "| pred conf", PRED_CONF)


## 5. Сабмит

`sample_submission.csv`. При `DO_SWEEP=1` — каталог `sweeps/` с комбинациями `WBF_SWEEP` и `CONF_SWEEP`.


In [ ]:
CONF_THR = float(os.environ.get("CONF_THR", "0.04"))
EMPTY_PRED = "0 0.000000 0 0 0 0"

CONF_SWEEP = [
    float(x.strip())
    for x in os.environ.get("CONF_SWEEP", "0.02,0.03,0.04,0.05,0.06").split(",")
    if x.strip()
]
DO_SWEEP = os.environ.get("DO_SWEEP", "0") == "1"


def _row_from_raw(raw: dict, conf_thr: float) -> str:
    xyxy = raw["xyxy"]
    cls = raw["cls"]
    conf = raw["conf"]
    orig_shape = raw["orig_shape"]

    if xyxy.shape[0] == 0:
        return EMPTY_PRED

    keep = conf >= conf_thr
    if not np.any(keep):
        return EMPTY_PRED

    xyxy = xyxy[keep].copy()
    cls = cls[keep]
    conf = conf[keep]
    order = np.argsort(-conf)
    xyxy, cls, conf = xyxy[order], cls[order], conf[order]

    im_h, im_w = orig_shape
    xyxy[:, [0, 2]] = np.clip(xyxy[:, [0, 2]], 0.0, im_w - 1.0)
    xyxy[:, [1, 3]] = np.clip(xyxy[:, [1, 3]], 0.0, im_h - 1.0)

    parts = []
    for i in range(len(cls)):
        x1, y1, x2, y2 = xyxy[i]
        if x2 < x1:
            x1, x2 = x2, x1
        if y2 < y1:
            y1, y2 = y2, y1
        parts += [
            str(int(cls[i])),
            f"{float(conf[i]):.6f}",
            f"{x1:.2f}",
            f"{y1:.2f}",
            f"{x2:.2f}",
            f"{y2:.2f}",
        ]
    return " ".join(parts) if parts else EMPTY_PRED


def build_submission(fused_map: dict[str, dict], conf_thr: float) -> pd.DataFrame:
    rows = []
    for stem, raw in fused_map.items():
        rows.append({"image_id": stem, "PredictionString": _row_from_raw(raw, conf_thr)})
    pred_df = pd.DataFrame(rows)
    template = pd.read_csv(SAMPLE_SUB)
    sub = template[["image_id"]].merge(pred_df, on="image_id", how="left")
    sub["PredictionString"] = sub["PredictionString"].fillna(EMPTY_PRED)
    sub.loc[sub["PredictionString"].str.strip() == "", "PredictionString"] = EMPTY_PRED
    return sub


base_sub = build_submission(results, CONF_THR)
out_path = ROOT / "sample_submission.csv"
base_sub.to_csv(out_path, index=False)
print("Saved base:", out_path, "rows:", len(base_sub), "| CONF_THR:", CONF_THR)

if DO_SWEEP:
    sweep_dir = ROOT / "sweeps"
    sweep_dir.mkdir(parents=True, exist_ok=True)
    summary = []
    for wbf_iou, fused_map in fused_by_wbf.items():
        for cth in CONF_SWEEP:
            sub = build_submission(fused_map, cth)
            fn = f"sample_submission_wbf{wbf_iou:.2f}_conf{cth:.2f}.csv"
            fp = sweep_dir / fn
            sub.to_csv(fp, index=False)
            summary.append({"file": fn, "wbf_iou": wbf_iou, "conf_thr": cth, "rows": len(sub)})

    summary_df = pd.DataFrame(summary).sort_values(["wbf_iou", "conf_thr"]).reset_index(drop=True)
    print("Sweep in", sweep_dir)
    summary_df.head(40)
else:
    print("DO_SWEEP=0 -> generated only sample_submission.csv")
